# ⚽ Predict the FIFA World Cup 2026

## 📖 Background

The 2026 FIFA World Cup is one of the biggest sporting events in the world, hosted across the United States, Canada, and Mexico. For the first time, the tournament expands to 48 teams, producing 104 matches across the group stage and knockout rounds.

Using machine learning, historical statistics, and soccer domain knowledge, predict match scores, corners, and cards for every fixture. You must submit all your predictions before a single ball is kicked.

The scoring system rewards precision: an exact scoreline earns maximum points, while close predictions still earn partial credit. Later rounds carry score multipliers, so a strong model that holds up in the knockout stages can leapfrog the competition. The challenge is designed to be difficult enough that no one can achieve a perfect score—even with AI assistance—but accessible enough that any data enthusiast can participate and score points.

## 💾 The data

You have access to the following files:

#### `data/group_fixtures.csv` — all 72 group stage matches
| Variable | Description |
|---|---|
| `match_id` | Unique match identifier |
| `group` | Group letter (A–L) |
| `home_team` | Home team name |
| `away_team` | Away team name |
| `date` | Match date (UTC) |
| `venue` | Stadium and city |

#### `data/knockout_slots.csv` — all 32 knockout round slots
| Variable | Description |
|---|---|
| `match_id` | Unique match identifier |
| `round` | Round name (e.g. `Quarter-final`) |
| `multiplier` | Score multiplier for this round |
| `slot_home` | Description of the home team slot (e.g. `Winner Group A`) |
| `slot_away` | Description of the away team slot |

| Variable | Description |
|---|---|

You may also bring in any external data—FIFA rankings, historical match results, player statistics—to build your predictions.

In [1]:
import pandas as pd


group_fixtures = pd.read_csv("C://Code Note//FIFA26//DataCamp Challenge//dataset//group_fixtures.csv")
group_fixtures.head()

,match_id,group,home_team,away_team,date_utc,venue
0,1,A,Mexico,South Africa,2026-06-11T19:00:00Z,"Estadio Azteca, Mexico City"
1,2,A,South Korea,UEFA Playoff D,2026-06-12T02:00:00Z,"Estadio Akron, Guadalajara"
2,3,B,Canada,UEFA Playoff A,2026-06-12T19:00:00Z,"BMO Field, Toronto"
3,4,D,USA,Paraguay,2026-06-13T01:00:00Z,"SoFi Stadium, Los Angeles"
4,5,D,Australia,UEFA Playoff C,2026-06-13T04:00:00Z,"BC Place, Vancouver"


In [2]:
knockout_slots = pd.read_csv("C://Code Note//FIFA26//DataCamp Challenge//dataset//knockout_slots.csv")
knockout_slots

,match_id,round,multiplier,date_utc,venue,slot_home,slot_away
0,73,Round of 32,1,2026-06-28T19:00:00Z,"SoFi Stadium, Los Angeles",Runner-up Group A,Runner-up Group B
1,74,Round of 32,1,2026-06-29T17:00:00Z,"NRG Stadium, Houston",Winner Group C,Runner-up Group F
2,75,Round of 32,1,2026-06-29T20:30:00Z,"Gillette Stadium, Boston",Winner Group E,Best 3rd (Groups A/B/C/D/F)
3,76,Round of 32,1,2026-06-30T01:00:00Z,"Estadio BBVA, Monterrey",Winner Group F,Runner-up Group C
4,77,Round of 32,1,2026-06-30T17:00:00Z,"AT&T Stadium, Dallas",Runner-up Group E,Runner-up Group I
5,78,Round of 32,1,2026-06-30T21:00:00Z,"MetLife Stadium, East Rutherford",Winner Group I,Best 3rd (Groups C/D/F/G/H)
6,79,Round of 32,1,2026-07-01T01:00:00Z,"Estadio Azteca, Mexico City",Winner Group A,Best 3rd (Groups C/E/F/H/I)
7,80,Round of 32,1,2026-07-01T16:00:00Z,"Mercedes-Benz Stadium, Atlanta",Winner Group L,Best 3rd (Groups E/H/I/J/K)
8,81,Round of 32,1,2026-07-01T20:00:00Z,"Lumen Field, Seattle",Winner Group G,Best 3rd (Groups A/E/H/I/J)
9,82,Round of 32,1,2026-07-02T00:00:00Z,"Levi's Stadium, Santa Clara",Winner Group D,Best 3rd (Groups B/E/F/I/J)


## 💪 Competition challenge

The 2026 World Cup has two phases:

- **Group stage** (matches 1–72): The 48 teams are split into 12 groups of 4. Every team plays the other 3 teams in their group once. The best teams from each group advance to the next phase.
- **Knockout stage** (matches 73–104): Single-elimination rounds — Round of 32, Round of 16, Quarter-finals, Semi-finals, and the Final. Lose once and you're out. Crucially, the two teams playing in each knockout match are not known in advance: they depend on who qualified from the group stage.

Submit predictions for **every match** in both phases. For each match you need to predict:

1. **Score** — the exact final scoreline (e.g. `2-1` means the home team scores 2, the away team scores 1). For knockout matches, the score is the result after 90 minutes and extra time — the penalty shootout is not included.
2. **Corners** — the number of corner kicks awarded in the match
3. **Yellow cards** — the number of yellow cards shown in the match
4. **Red cards** — the number of red cards shown in the match

For **group stage** matches, also predict:
- **Winning team** — which team wins the individual match (use `home`, `away`, or `draw`)

For **knockout round** matches, also predict:
- **Matchup** — which two teams you predict will be playing in that slot. Because the bracket is determined by group stage results, you need to predict which teams advance far enough to meet in each round.
- **Match winner** — which team wins the match (use `home` or `away`)
- **Penalties** — whether the match goes to a penalty shootout (`True` or `False`)

### Scoring system

| Category | Condition | Points |
|---|---|---|
| Score | Exact scoreline | 25 |
| Score | Correct goal difference, wrong score | 10 |
| Score | Correct total goals, wrong score | 10 |
| Corners | Exact number | 10 |
| Corners | Off by 2 | 5 |
| Yellow cards | Exact number | 10 |
| Yellow cards | Off by 1 | 5 |
| Red cards | Exact number | 5 |
| Winning team *(group stage only)* | Correct | 40 |
| Matchup *(knockout only)* | Both teams correct | 20 |
| Matchup *(knockout only)* | One team correct | 10 |
| Match winner *(knockout only)* | Correct | 20 |
| Penalties *(knockout only)* | Correct | 5 |

All points for a match are multiplied by the round factor:

| Round | Multiplier |
|---|---|
| Group stage | ×1 |
| Round of 32 | ×1 |
| Round of 16 | ×2 |
| Quarter-final | ×4 |
| Semi-final | ×8 |
| Third-place playoff | ×8 |
| Final | ×16 |

## 🗓️ Group stage predictions

Fill in your predictions for all 72 group stage matches below.

In [3]:
group_predictions = group_fixtures.copy()

# Fill in your predictions for each match
# Example (match 1 — Mexico vs South Africa): predicted_home_goals=2, predicted_away_goals=1, corners=9, yellow_cards=3, red_cards=0, winning_team='home'
group_predictions['predicted_home_goals'] = None   # e.g. 2
group_predictions['predicted_away_goals'] = None   # e.g. 1
group_predictions['corners']              = None   # e.g. 9
group_predictions['yellow_cards']         = None   # e.g. 3
group_predictions['red_cards']            = None   # e.g. 0
group_predictions['winning_team']         = None   # "home", "away", or "draw"

group_predictions

,match_id,group,home_team,away_team,date_utc,venue,predicted_home_goals,predicted_away_goals,corners,yellow_cards,red_cards,winning_team
0,1,A,Mexico,South Africa,2026-06-11T19:00:00Z,"Estadio Azteca, Mexico City",None,None,None,None,None,None
1,2,A,South Korea,UEFA Playoff D,2026-06-12T02:00:00Z,"Estadio Akron, Guadalajara",None,None,None,None,None,None
2,3,B,Canada,UEFA Playoff A,2026-06-12T19:00:00Z,"BMO Field, Toronto",None,None,None,None,None,None
3,4,D,USA,Paraguay,2026-06-13T01:00:00Z,"SoFi Stadium, Los Angeles",None,None,None,None,None,None
4,5,D,Australia,UEFA Playoff C,2026-06-13T04:00:00Z,"BC Place, Vancouver",None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...
67,68,L,Croatia,Ghana,2026-06-27T21:00:00Z,"Lincoln Financial Field, Philadelphia",None,None,None,None,None,None
68,69,K,Colombia,Portugal,2026-06-27T23:30:00Z,"Hard Rock Stadium, Miami",None,None,None,None,None,None
69,70,K,FIFA Playoff 1,Uzbekistan,2026-06-27T23:30:00Z,"Mercedes-Benz Stadium, Atlanta",None,None,None,None,None,None
70,71,J,Algeria,Austria,2026-06-28T02:00:00Z,"GEHA Field at Arrowhead Stadium, Kansas City",None,None,None,None,None,None


## 🏆 Knockout stage predictions

For knockout matches you also predict **which teams are playing**. Fill in the team names based on your group stage predictions, then add your match predictions.

In [4]:
knockout_predictions = knockout_slots.copy()

# Fill in your predictions for each knockout match
# Example (match 73 — Round of 32): predicted_home_team='Brazil', predicted_away_team='France', predicted_home_goals=1, predicted_away_goals=0, corners=8, yellow_cards=2, red_cards=0, match_winner='home', penalties=False
knockout_predictions['predicted_home_team']  = None   # e.g. "Brazil"
knockout_predictions['predicted_away_team']  = None   # e.g. "France"
knockout_predictions['predicted_home_goals'] = None   # e.g. 1
knockout_predictions['predicted_away_goals'] = None   # e.g. 0
knockout_predictions['corners']              = None   # e.g. 8
knockout_predictions['yellow_cards']         = None   # e.g. 2
knockout_predictions['red_cards']            = None   # e.g. 0
knockout_predictions['match_winner']         = None   # "home" or "away"
knockout_predictions['penalties']            = None   # True or False

knockout_predictions

,match_id,round,multiplier,date_utc,venue,slot_home,slot_away,predicted_home_team,predicted_away_team,predicted_home_goals,predicted_away_goals,corners,yellow_cards,red_cards,match_winner,penalties
0,73,Round of 32,1,2026-06-28T19:00:00Z,"SoFi Stadium, Los Angeles",Runner-up Group A,Runner-up Group B,None,None,None,None,None,None,None,None,None
1,74,Round of 32,1,2026-06-29T17:00:00Z,"NRG Stadium, Houston",Winner Group C,Runner-up Group F,None,None,None,None,None,None,None,None,None
2,75,Round of 32,1,2026-06-29T20:30:00Z,"Gillette Stadium, Boston",Winner Group E,Best 3rd (Groups A/B/C/D/F),None,None,None,None,None,None,None,None,None
3,76,Round of 32,1,2026-06-30T01:00:00Z,"Estadio BBVA, Monterrey",Winner Group F,Runner-up Group C,None,None,None,None,None,None,None,None,None
4,77,Round of 32,1,2026-06-30T17:00:00Z,"AT&T Stadium, Dallas",Runner-up Group E,Runner-up Group I,None,None,None,None,None,None,None,None,None
5,78,Round of 32,1,2026-06-30T21:00:00Z,"MetLife Stadium, East Rutherford",Winner Group I,Best 3rd (Groups C/D/F/G/H),None,None,None,None,None,None,None,None,None
6,79,Round of 32,1,2026-07-01T01:00:00Z,"Estadio Azteca, Mexico City",Winner Group A,Best 3rd (Groups C/E/F/H/I),None,None,None,None,None,None,None,None,None
7,80,Round of 32,1,2026-07-01T16:00:00Z,"Mercedes-Benz Stadium, Atlanta",Winner Group L,Best 3rd (Groups E/H/I/J/K),None,None,None,None,None,None,None,None,None
8,81,Round of 32,1,2026-07-01T20:00:00Z,"Lumen Field, Seattle",Winner Group G,Best 3rd (Groups A/E/H/I/J),None,None,None,None,None,None,None,None,None
9,82,Round of 32,1,2026-07-02T00:00:00Z,"Levi's Stadium, Santa Clara",Winner Group D,Best 3rd (Groups B/E/F/I/J),None,None,None,None,None,None,None,None,None


## ✅ Checklist before publishing into the competition

- Rename your workspace to make it descriptive of your work. N.B. you should leave the notebook name as `notebook.ipynb`.
- Remove redundant cells like the judging criteria, so the workbook is focused on your predictions.
- Make sure all prediction cells are filled in—`None` values will score 0 points.
- Check that all cells run without error.
- Make sure your workbook is published before **June 10, 2026 at 09:00 UTC**.

## ⏳ Time is ticking. Good luck!

In [5]:
from math import exp, factorial
from itertools import product

In [6]:
HOME_ADVANTAGE   = 1.10   
MAX_GOALS        = 10    
ELO_SCALE        = 1800  
AVG_GOALS_TOTAL  = 2.55

In [7]:
group_fixtures.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 72 entries, 0 to 71
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   match_id   72 non-null     int64 
 1   group      72 non-null     object
 2   home_team  72 non-null     object
 3   away_team  72 non-null     object
 4   date_utc   72 non-null     object
 5   venue      72 non-null     object
dtypes: int64(1), object(5)
memory usage: 3.5+ KB


In [8]:
group_fixtures.shape

(72, 6)

In [9]:
home_team =group_fixtures.home_team.unique()

In [10]:
away_team = group_fixtures.away_team.unique()

In [11]:
team = []
for i in range(len(home_team)):
    if home_team[i] not in team:
        team.append(home_team[i])
    elif away_team[i] not in team:
        team.append(away_team[i])

In [12]:
print(team)

['Mexico', 'South Korea', 'Canada', 'USA', 'Australia', 'Qatar', 'Brazil', 'Haiti', 'Germany', 'Netherlands', "Côte d'Ivoire", 'UEFA Playoff B', 'Spain', 'Belgium', 'Saudi Arabia', 'Iran', 'Austria', 'France', 'FIFA Playoff 2', 'Argentina', 'Portugal', 'England', 'Ghana', 'Uzbekistan', 'UEFA Playoff D', 'Switzerland', 'UEFA Playoff C', 'Scotland', 'Tunisia', 'Ecuador', 'Uruguay', 'New Zealand', 'Norway', 'Jordan', 'Panama', 'Colombia', 'UEFA Playoff A', 'Morocco', 'South Africa', 'Curaçao', 'Japan', 'Paraguay', 'Senegal', 'Cabo Verde', 'Egypt', 'Croatia', 'FIFA Playoff 1', 'Algeria']


In [13]:
if 'Switzerland' in team:
    print("Switzerland is in the tournament!")

Switzerland is in the tournament!


In [14]:
group_fixtures[group_fixtures.home_team == 'FIFA Playoff 1']

,match_id,group,home_team,away_team,date_utc,venue
69,70,K,FIFA Playoff 1,Uzbekistan,2026-06-27T23:30:00Z,"Mercedes-Benz Stadium, Atlanta"


UEFA Playoff B - Sweden  
UEFA Playoff A - Bosnia and Herzegovina   
FIFA Playoff 1 - Congo DR    
FIFA Playoff 2 - Iraq

### Updating the team 

In [15]:
group_fixtures.replace('UEFA Playoff B', 'Sweden', inplace=True)
group_fixtures.replace('UEFA Playoff A', 'Bosnia and Herzegovina', inplace=True)
group_fixtures.replace('UEFA Playoff D', 'Czech Republic', inplace=True)
group_fixtures.replace('UEFA Playoff C', 'Turkey', inplace=True)
group_fixtures.replace('FIFA Playoff 1', 'Congo DR', inplace=True)
group_fixtures.replace('FIFA Playoff 2', 'Iraq', inplace=True)
group_fixtures.replace("Côte d'Ivoire", "Ivory Coast", inplace=True)
group_fixtures.replace('Cabo Verde', 'Cape Verde', inplace=True)
group_fixtures.replace('Congo DR', 'DR Congo', inplace=True)

Now I have full team participating list updated.

In [16]:
TEAMS = {
    # ── GROUP A ──────────────────────────────────────────────
    "Mexico": {
        "elo": 1750, "attack": 1.6, "defense": 1.1,
        "confederation": "CONCACAF", "group": "A"
    },
    "South Africa": {
        "elo": 1480, "attack": 1.0, "defense": 1.4,
        "confederation": "CAF", "group": "A"
    },
    "South Korea": {
        "elo": 1620, "attack": 1.4, "defense": 1.1,
        "confederation": "AFC", "group": "A"
    },
    "Czech Republic": {
        "elo": 1590, "attack": 1.3, "defense": 1.2,
        "confederation": "UEFA", "group": "A"
    },
 
    # ── GROUP B ──────────────────────────────────────────────
    "Canada": {
        "elo": 1640, "attack": 1.5, "defense": 1.2,
        "confederation": "CONCACAF", "group": "B"
    },
    "Bosnia and Herzegovina": {
        "elo": 1540, "attack": 1.3, "defense": 1.3,
        "confederation": "UEFA", "group": "B"
    },
    "Qatar": {
        "elo": 1440, "attack": 0.9, "defense": 1.5,
        "confederation": "AFC", "group": "B"
    },
    "Switzerland": {
        "elo": 1720, "attack": 1.6, "defense": 1.0,
        "confederation": "UEFA", "group": "B"
    },
 
    # ── GROUP C ──────────────────────────────────────────────
     "Brazil": {
        "elo": 2050, "attack": 2.1, "defense": 0.8,
        "confederation": "CONMEBOL", "group": "C"
    },
    "Morocco": {
        "elo": 1730, "attack": 1.4, "defense": 0.8,
        "confederation": "CAF", "group": "C"
    },
    "Scotland": {
        "elo": 1640, "attack": 1.4, "defense": 1.2,
        "confederation": "UEFA", "group": "C"
    },
    "Haiti": {
    "elo": 1420, "attack": 0.9, "defense": 1.4,
    "confederation": "CONCACAF", "group": "C"
    },
 
    # ── GROUP D ──────────────────────────────────────────────
    "USA": {
        "elo": 1710, "attack": 1.5, "defense": 1.1,
        "confederation": "CONCACAF", "group": "D"
    },
    "Paraguay": {
        "elo": 1520, "attack": 1.1, "defense": 1.3,
        "confederation": "CONMEBOL", "group": "D"
    },
    "Australia": {
        "elo": 1590, "attack": 1.3, "defense": 1.2,
        "confederation": "AFC", "group": "D"
    },
    "Turkey": {
        "elo": 1650, "attack": 1.5, "defense": 1.2,
        "confederation": "UEFA", "group": "D"
    },
 
    # ── GROUP E ──────────────────────────────────────────────
    "Germany": {
        "elo": 1970, "attack": 2.0, "defense": 0.9,
        "confederation": "UEFA", "group": "E"
    },
    "Curaçao": {
        "elo": 1380, "attack": 0.8, "defense": 1.6,
        "confederation": "CONCACAF", "group": "E"
    },
    "Ecuador": {
        "elo": 1560, "attack": 1.2, "defense": 1.2,
        "confederation": "CONMEBOL", "group": "E"
    },
    "Ivory Coast": {
        "elo": 1620, "attack": 1.4, "defense": 1.2,
        "confederation": "CAF", "group": "E"
    },
 
    # ── GROUP F ──────────────────────────────────────────────
    "Netherlands": {
        "elo": 1920, "attack": 1.9, "defense": 0.9,
        "confederation": "UEFA", "group": "F"
    },
    "Japan": {
        "elo": 1760, "attack": 1.6, "defense": 0.9,
        "confederation": "AFC", "group": "F"
    },
    "Tunisia": {
        "elo": 1530, "attack": 1.2, "defense": 1.3,
        "confederation": "CAF", "group": "F"
    },
    "Sweden": {
        "elo": 1660, "attack": 1.5, "defense": 1.1,
        "confederation": "UEFA", "group": "F"
    },
 
    # ── GROUP G ──────────────────────────────────────────────
    "Belgium": {
        "elo": 1880, "attack": 1.9, "defense": 0.9,
        "confederation": "UEFA", "group": "G"
    },
    "Egypt": {
        "elo": 1600, "attack": 1.3, "defense": 1.2,
        "confederation": "CAF", "group": "G"
    },
    "Iran": {
        "elo": 1600, "attack": 1.2, "defense": 1.1,
        "confederation": "AFC", "group": "G"
    },
    "New Zealand": {
       "elo": 1380, "attack": 0.8, "defense": 1.3,
       "confederation": "OFC", "group": "G"
    },
 
    # ── GROUP H ──────────────────────────────────────────────
    "Spain": {
        "elo": 2100, "attack": 2.1, "defense": 0.7,
        "confederation": "UEFA", "group": "H"
    },
    "Cape Verde": {
        "elo": 1460, "attack": 1.0, "defense": 1.3,
        "confederation": "CAF", "group": "H"
    },
    "Uruguay": {
        "elo": 1780, "attack": 1.7, "defense": 0.9,
        "confederation": "CONMEBOL", "group": "H"
    },
    "Saudi Arabia": {
            "elo": 1565, "attack": 1.1, "defense": 1.2,
            "confederation": "AFC", "group": "H"
    },
 
    # ── GROUP I ──────────────────────────────────────────────
    "France": {
        "elo": 2060, "attack": 2.1, "defense": 0.8,
        "confederation": "UEFA", "group": "I"
    },
    "Senegal": {
        "elo": 1680, "attack": 1.4, "defense": 1.0,
        "confederation": "CAF", "group": "I"
    },
    "Iraq": {
        "elo": 1430, "attack": 1.0, "defense": 1.5,
        "confederation": "AFC", "group": "I"
    },
    
    "Norway": {
        "elo": 1700, "attack": 1.8, "defense": 1.1,
        "confederation": "UEFA", "group": "I"
    },
 
    # ── GROUP J ──────────────────────────────────────────────
    "Argentina": {
        "elo": 2080, "attack": 2.2, "defense": 0.8,
        "confederation": "CONMEBOL", "group": "J"
    },
    "Algeria": {
        "elo": 1560, "attack": 1.2, "defense": 1.3,
        "confederation": "CAF", "group": "J"
    },
    "Austria": {
        "elo": 1720, "attack": 1.7, "defense": 1.0,
        "confederation": "UEFA", "group": "J"
    },
    "Jordan": {
        "elo": 1420, "attack": 0.9, "defense": 1.5,
        "confederation": "AFC", "group": "J"
    },
 
    # ── GROUP K ──────────────────────────────────────────────
    "Portugal": {
        "elo": 1970, "attack": 2.0, "defense": 0.9,
        "confederation": "UEFA", "group": "K"
    },
    "DR Congo": {
        "elo": 1450, "attack": 1.0, "defense": 1.4,
        "confederation": "CAF", "group": "K"
    },
    "Colombia": {
        "elo": 1730, "attack": 1.7, "defense": 1.0,
        "confederation": "CONMEBOL", "group": "K"
    },
    "Uzbekistan": {
        "elo": 1410, "attack": 0.9, "defense": 1.5,
        "confederation": "AFC", "group": "K"
    },
 
    # ── GROUP L ──────────────────────────────────────────────
    "England": {
        "elo": 2030, "attack": 2.0, "defense": 0.8,
        "confederation": "UEFA", "group": "L"
    },
    "Croatia": {
        "elo": 1820, "attack": 1.7, "defense": 0.9,
        "confederation": "UEFA", "group": "L"
    },
    "Ghana": {
    "elo": 1510, "attack": 1.1, "defense": 1.3,
    "confederation": "CAF", "group": "L"
    },
    "Panama": {
    "elo": 1470, "attack": 1.0, "defense": 1.3,
    "confederation": "CONCACAF", "group": "L"
    }
    
}

In [17]:
rank = pd.read_csv("C:\\Code Note\\FIFA26\\DataCamp Challenge\\result\\fifa_rankings.csv")

In [18]:
rank

,Rank,Team Name,Points,Previous Rank,Country Code
0,1.0,Brazil,1684.97,1.0,BRA
1,2.0,Portugal,1575.32,2.0,POR
2,3.0,Spain,1574.39,3.0,ESP
3,4.0,Argentina,1513.38,4.0,ARG
4,5.0,IR Iran,1507.71,5.0,IRN
...,...,...,...,...,...
206,NaN,Tunisia,1021.22,NaN,TUN
207,NaN,Rwanda,1000.00,NaN,RWA
208,NaN,Zimbabwe,1000.00,NaN,ZIM
209,NaN,Bahamas,1000.00,NaN,BAH


In [19]:
for i, item in rank.iterrows():
    team_name = item['Team Name']
    if team_name in TEAMS:
        TEAMS[team_name]['elo'] = item['Points']
        print(f"Updated {team_name} Elo rating to {item['Points']}")

Updated Brazil Elo rating to 1684.97
Updated Portugal Elo rating to 1575.32
Updated Spain Elo rating to 1574.39
Updated Argentina Elo rating to 1513.38
Updated Morocco Elo rating to 1486.53
Updated France Elo rating to 1397.25
Updated Japan Elo rating to 1315.11
Updated Croatia Elo rating to 1312.03
Updated Paraguay Elo rating to 1236.37
Updated Uzbekistan Elo rating to 1187.74
Updated Iraq Elo rating to 1186.9
Updated Colombia Elo rating to 1179.06
Updated Uruguay Elo rating to 1171.13
Updated Netherlands Elo rating to 1164.89
Updated Panama Elo rating to 1138.15
Updated Belgium Elo rating to 1136.09
Updated Egypt Elo rating to 1084.16
Updated Saudi Arabia Elo rating to 1083.52
Updated Germany Elo rating to 1077.31
Updated Bosnia and Herzegovina Elo rating to 1067.17
Updated New Zealand Elo rating to 1057.64
Updated Australia Elo rating to 1028.85
Updated South Africa Elo rating to 1004.53
Updated Sweden Elo rating to 1001.1
Updated Canada Elo rating to 986.41
Updated Curaçao Elo rati

In [20]:
def get_team(name: str) -> dict:

    if name not in TEAMS:
        raise ValueError(f"Team '{name}' not found in TEAMS dictionary.")
    
    return TEAMS[name]

def get_team_elo(name: str) -> int:
    team_info = get_team(name)
    return team_info['elo']

def get_team_group(name: str) -> str:
    team_info = get_team(name)
    return team_info['group']

def elo_lambda(elo_a: int, elo_b: int) -> float:

    diff = elo_b - elo_a
    return 10 ** (diff / ELO_SCALE)

In [21]:
f = get_team_group("France")
print(f)

I


In [22]:
def compute_lambda(home_team: str, away_team: str) -> float:
    elo_a = get_team_elo(home_team)
    elo_b = get_team_elo(away_team)
    a = get_team(home_team)
    b = get_team(away_team)
    base_lambda = AVG_GOALS_TOTAL / 2
    
    lambda_a = base_lambda * a['attack'] * a['defense'] * elo_lambda(elo_a, elo_b) * HOME_ADVANTAGE
    lambda_b = base_lambda * b['attack'] * b['defense'] * (1/ elo_lambda(elo_b, elo_a))

    return lambda_a, lambda_b

def poisson_dis(lam: float, k:int) -> float:
    if lam <= 0:
        return 0.0
    return (lam ** k * exp(-lam)) / factorial(k)

def score_line_matrix(lambda_a: float, lambda_b: float):
    matrix = {}
    for goal_a, goal_b in product(range(MAX_GOALS + 1), repeat=2):
        prob_a = poisson_dis(lambda_a, goal_a)
        prob_b = poisson_dis(lambda_b, goal_b)
        matrix[(goal_a, goal_b)] = prob_a * prob_b

    return matrix

def match_outcome_probabilities(home_team: str, away_team:str, neutral: bool = True) -> dict:

    group = get_team_group(home_team)
    lam_a, lam_b = compute_lambda(home_team, away_team)
    matrix = score_line_matrix(lam_a, lam_b)
    
    prob_home_win = sum(prob for (a, b), prob in matrix.items() if a > b)
    prob_away_win = sum(prob for (a, b), prob in matrix.items() if a < b)
    prob_draw     = sum(prob for (a, b), prob in matrix.items() if a == b)

    top = sorted(matrix.items(), key=lambda x: x[1], reverse=True)[:5]
    top_scorelines = [(f"{ga}-{gb}", round(p * 100, 2)) for (ga, gb), p in top]
    return {
        "home_team": home_team,
        "away_team": away_team,
        "Group": group,
        "home_win": prob_home_win*100,
        "away_win": prob_away_win*100,
        "draw": prob_draw*100,
        "top_scorelines": top_scorelines[0:3],
        "expected_total_goals": round(lam_a + lam_b, 2),
        "Probable team win": home_team if prob_home_win > prob_away_win else away_team if prob_away_win > prob_home_win else "Draw"
        
    }



In [23]:
team = []
for x, row in group_fixtures.iterrows():
    home = row['home_team']
    away = row['away_team']
    probs = match_outcome_probabilities(home, away, True)
    print(probs)
    team.append(probs)
    # # print(f"{home} vs {away}: {probs}")

{'home_team': 'Mexico', 'away_team': 'South Africa', 'Group': 'A', 'home_win': 53.77970613664416, 'away_win': 27.77449835902443, 'draw': 18.43621952862804, 'top_scorelines': [('2-1', 7.14), ('2-2', 6.75), ('3-1', 6.22)], 'expected_total_goals': 4.5, 'Probable team win': 'Mexico'}
{'home_team': 'South Korea', 'away_team': 'Czech Republic', 'Group': 'A', 'home_win': 42.83974749548807, 'away_win': 36.49801034719543, 'draw': 20.66050589733746, 'top_scorelines': [('2-1', 7.63), ('1-1', 7.34), ('2-2', 7.3)], 'expected_total_goals': 3.99, 'Probable team win': 'South Korea'}
{'home_team': 'Canada', 'away_team': 'Bosnia and Herzegovina', 'Group': 'B', 'home_win': 48.085270574939706, 'away_win': 34.15828935510981, 'draw': 17.73596836816435, 'top_scorelines': [('2-2', 6.24), ('3-2', 5.82), ('2-1', 5.22)], 'expected_total_goals': 5.19, 'Probable team win': 'Canada'}
{'home_team': 'USA', 'away_team': 'Paraguay', 'Group': 'D', 'home_win': 52.76681951664099, 'away_win': 30.80337117386605, 'draw': 16.

In [25]:
df = pd.DataFrame(team)
df.head()
df.to_csv('C://Code Note//FIFA26//DataCamp Challenge//result//predict_matches.csv')

In [26]:
df.groupby('Group').apply(lambda x: x.sort_index())

C:\Users\sanja\AppData\Local\Temp\ipykernel_22940\1237485375.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df.groupby('Group').apply(lambda x: x.sort_index())


home_team       away_team Group   home_win   away_win  \
Group                                                                  
A     0           Mexico    South Africa     A  53.779706  27.774498   
      1      South Korea  Czech Republic     A  42.839747  36.498010   
      24  Czech Republic    South Africa     A  39.191799  28.973966   
      27          Mexico     South Korea     A  54.949222  29.245212   
      52  Czech Republic          Mexico     A  33.818633  35.136383   
...                  ...             ...   ...        ...        ...   
L     22           Ghana          Panama     L  50.881052  32.839922   
      45         England           Ghana     L  45.723934  31.470084   
      46          Panama         Croatia     L  37.706431  43.390236   
      66          Panama         England     L  34.053475  41.330682   
      67         Croatia           Ghana     L  38.455281  30.135285   

               draw                              top_scorelines  \
Group                                                             
A     0   18.436220     [(2-1, 7.14), (2-2, 6.75), (3-1, 6.22)]   
      1   20.660506      [(2-1, 7.63), (1-1, 7.34), (2-2, 7.3)]   
      24  31.834234  [(1-0, 15.81), (0-0, 15.28), (1-1, 13.34)]   
      27  11.794401     [(5-4, 3.14), (6-4, 3.01), (5-5, 2.87)]   
      52  31.044982  [(0-1, 13.85), (0-0, 13.82), (1-1, 13.53)]   
...             ...                                         ...   
L     22  16.201017     [(3-2, 5.36), (3-3, 4.88), (2-2, 4.87)]   
      45  22.805617    [(1-1, 10.05), (2-1, 9.07), (1-2, 7.37)]   
      46  18.895712      [(2-2, 6.9), (1-2, 6.06), (2-1, 5.67)]   
      66  24.615749    [(1-1, 11.48), (1-2, 8.83), (0-1, 8.35)]   
      67  31.409432  [(1-0, 15.16), (0-0, 14.55), (1-1, 13.43)]   

          expected_total_goals Probable team win  
Group                                             
A     0                   4.50            Mexico  
      1                   3.99       South Korea  
      24                  1.88    Czech Republic  
      27                 10.31            Mexico  
      52                  1.98            Mexico  
...                        ...               ...  
L     22                  6.04             Ghana  
      45                  3.27           England  
      46                  4.71           Croatia  
      66                  2.91           England  
      67                  1.93           Croatia  

[72 rows x 9 columns]

In [27]:
rows = []

for _, row in df.iterrows():
    group = row["Group"]

    if row["home_win"] > row["away_win"]:        # home wins
        rows.append({"team": row["home_team"], "group": group, "W":1, "D":0, "L":0, "pts":3})
        rows.append({"team": row["away_team"], "group": group, "W":0, "D":0, "L":1, "pts":0})

    elif row["away_win"] > row["home_win"]:       # away wins
        rows.append({"team": row["home_team"], "group": group, "W":0, "D":0, "L":1, "pts":0})
        rows.append({"team": row["away_team"], "group": group, "W":1, "D":0, "L":0, "pts":3})

    else:                                          # draw
        rows.append({"team": row["home_team"], "group": group, "W":0, "D":1, "L":0, "pts":1})
        rows.append({"team": row["away_team"], "group": group, "W":0, "D":1, "L":0, "pts":1})

points_df = pd.DataFrame(rows)

# Aggregate per team
table = (points_df
         .groupby(["group", "team"])
         .sum()
         .reset_index()
         .sort_values(["group", "pts"], ascending=[True, False]))

print(table)

   group                    team  W  D  L  pts
1      A                  Mexico  3  0  0    9
0      A          Czech Republic  1  0  2    3
2      A            South Africa  1  0  2    3
3      A             South Korea  1  0  2    3
5      B                  Canada  3  0  0    9
7      B             Switzerland  2  0  1    6
4      B  Bosnia and Herzegovina  1  0  2    3
6      B                   Qatar  0  0  3    0
11     C                Scotland  3  0  0    9
8      C                  Brazil  2  0  1    6
9      C                   Haiti  1  0  2    3
10     C                 Morocco  0  0  3    0
15     D                     USA  3  0  0    9
14     D                  Turkey  2  0  1    6
13     D                Paraguay  1  0  2    3
12     D               Australia  0  0  3    0
18     E                 Germany  3  0  0    9
19     E             Ivory Coast  2  0  1    6
17     E                 Ecuador  1  0  2    3
16     E                 Curaçao  0  0  3    0
21     F     

In [151]:
table.to_csv("group_stage_predictions.csv", index=False)